# 🌸 Manga Translator Studio - AI Backend (lama-manga-onnx & Manga-OCR)
> **Google Colab GPU Backend Server** tích hợp mô hình **lama-manga-onnx** chuyên dụng xóa chữ tranh Manga/Comic siêu sạch và **Manga-OCR** nhận diện tiếng Nhật không kiểm duyệt.

### 📋 Hướng dẫn sử dụng (1-Click Run):
1. Nhấn nút **Play ▶️ (Run cell)** ở ô code bên dưới.
2. Đợi khoảng 1-2 phút để hệ thống tự động cài đặt thư viện và nạp mô hình **lama-manga-onnx** vào GPU.
3. Khi hoàn tất, copy đường link URL (dạng `https://xxxx.trycloudflare.com` hoặc `https://xxxx.ngrok-free.app`) được in ra ở cuối.
4. Dán URL này vào nút **Colab GPU** trên Web App hoặc APK điện thoại để bắt đầu dịch truyện!



In [ ]:
#@title 🚀 Khởi chạy Manga Translator AI Server (lama-manga-onnx & Manga-OCR) { display-mode: "form" }
#@markdown Nhập Ngrok Token nếu muốn dùng Ngrok ổn định (để trống sẽ tự động dùng Cloudflare Tunnel miễn phí không cần tài khoản):
NGROK_AUTH_TOKEN = "" #@param {type:"string"}

import os
import sys
import time
import subprocess
import threading
import urllib.request
import re
import io
import json
import base64
from typing import List, Optional, Dict, Any

print("="*65)
print("⏳ [1/4] Đang cài đặt thư viện (ONNX Runtime GPU, LaMa Manga, Manga-OCR)...")
print("="*65)

# Cài đặt thư viện cần thiết
packages = [
    "fastapi",
    "uvicorn",
    "pydantic",
    "pyngrok",
    "deep-translator",
    "manga-ocr",
    "pillow",
    "numpy",
    "google-generativeai",
    "onnxruntime-gpu"
]
subprocess.run(["pip", "install", "-q"] + packages, check=False)

import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from deep_translator import GoogleTranslator
from pyngrok import ngrok
import onnxruntime as ort

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 1. Nạp mô hình LaMa Manga ONNX (lama-manga-onnx)
class LamaMangaOnnx:
    def __init__(self, dev="cuda"):
        model_path = "lama-manga.onnx"
        if not os.path.exists(model_path) or os.path.getsize(model_path) < 10000000:
            print("⏳ [2/4] Đang tải mô hình lama-manga-onnx (~200MB)... Vui lòng đợi trong giây lát...")
            url = "https://huggingface.co/mayocream/lama-manga-onnx/resolve/main/lama-manga.onnx"
            urllib.request.urlretrieve(url, model_path)
            print("✅ Đã tải mô hình lama-manga-onnx thành công!")
        
        print("⏳ Đang nạp mô hình lama-manga-onnx vào ONNX Runtime...")
        available = ort.get_available_providers()
        providers = []
        if dev == "cuda" and "CUDAExecutionProvider" in available:
            providers.append("CUDAExecutionProvider")
        if "CPUExecutionProvider" in available:
            providers.append("CPUExecutionProvider")
        if not providers:
            providers = available
            
        print(f"ONNX Execution Providers: {providers}")
        self.session = ort.InferenceSession(model_path, providers=providers)
        self.input_names = [inp.name for inp in self.session.get_inputs()]
        self.output_name = self.session.get_outputs()[0].name
        print(f"✅ lama-manga-onnx sẵn sàng! (Inputs: {self.input_names}, Output: {self.output_name})")

    def __call__(self, img: Image.Image, mask: Image.Image) -> Image.Image:
        orig_w, orig_h = img.size
        mod_w = max(8, (orig_w // 8) * 8)
        mod_h = max(8, (orig_h // 8) * 8)
        
        img_resized = img.resize((mod_w, mod_h), Image.Resampling.BILINEAR)
        mask_resized = mask.resize((mod_w, mod_h), Image.Resampling.NEAREST).convert("L")

        img_np = np.array(img_resized).astype(np.float32) / 255.0
        img_np = np.transpose(img_np, (2, 0, 1))
        img_np = np.expand_dims(img_np, axis=0)

        mask_np = (np.array(mask_resized).astype(np.float32) / 255.0 > 0.5).astype(np.float32)
        mask_np = np.expand_dims(mask_np, axis=(0, 1))

        inputs = {
            self.input_names[0]: img_np,
            self.input_names[1]: mask_np
        }
        outputs = self.session.run([self.output_name], inputs)
        res_np = outputs[0][0]
        if res_np.shape[0] == 3:
            res_np = np.transpose(res_np, (1, 2, 0))

        res_np = np.clip(res_np * 255.0, 0, 255).astype(np.uint8)
        return Image.fromarray(res_np).resize((orig_w, orig_h), Image.Resampling.BILINEAR)

lama = LamaMangaOnnx(dev=device)

# 2. Nạp Manga-OCR cho tiếng Nhật
mocr = None
try:
    print("⏳ [3/4] Đang nạp Manga-OCR (Chế độ Uncensored)...")
    from manga_ocr import MangaOcr
    mocr = MangaOcr(force_cpu=True)
    print("✅ Manga-OCR sẵn sàng!")
except Exception as me:
    print(f"⚠️ Manga-OCR note: {me}")

# 3. Khởi tạo FastAPI Server
app = FastAPI(title="Manga Translator AI Server - lama-manga-onnx")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

def b64_to_pil(b64_str: str) -> Image.Image:
    if "," in b64_str: b64_str = b64_str.split(",")[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str))).convert("RGB")

def pil_to_b64(pil_img: Image.Image, fmt="PNG") -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format=fmt)
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

class InpaintReq(BaseModel):
    imageBase64: str
    maskBase64: str
class OcrReq(BaseModel):
    imageBase64: str
    boxes: List[Dict[str, Any]]
class TransReq(BaseModel):
    texts: List[str]
    sourceLang: Optional[str] = "auto"
    targetLang: Optional[str] = "vi"
    engine: Optional[str] = "google"
    apiKey: Optional[str] = None

@app.get("/health")
def health():
    return {
        "status": "online",
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "model": "lama-manga-onnx",
        "lama_ready": True,
        "mocr_ready": mocr is not None
    }

@app.post("/api/inpaint")
async def do_inpaint(req: InpaintReq):
    try:
        img = b64_to_pil(req.imageBase64)
        mask = b64_to_pil(req.maskBase64).convert("L")
        res = lama(img, mask)
        return {"success": True, "cleanedImageBase64": pil_to_b64(res)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/ocr")
async def do_ocr(req: OcrReq):
    try:
        img = b64_to_pil(req.imageBase64)
        results = []
        for box in req.boxes:
            x, y = max(0, int(box.get("x", 0))), max(0, int(box.get("y", 0)))
            w, h = max(10, int(box.get("width", 50))), max(10, int(box.get("height", 50)))
            crop = img.crop((x, y, min(img.width, x + w), min(img.height, y + h)))
            text = mocr(crop) if mocr is not None else "(OCR Text)"
            results.append({"box": box, "text": text})
        return {"success": True, "results": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/translate")
async def do_translate(req: TransReq):
    try:
        if req.engine == "gemini" and req.apiKey:
            import google.generativeai as genai
            genai.configure(api_key=req.apiKey)
            model = genai.GenerativeModel("gemini-2.0-flash")
            prompt = f"Dịch các câu manga sau sang tiếng Việt:\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(req.texts))
            res = model.generate_content(prompt)
            m = re.search(r'\[.*\]', res.text, re.DOTALL)
            out = json.loads(m.group(0)) if m else [res.text.strip()]
        else:
            translator = GoogleTranslator(source=req.sourceLang or "auto", target=req.targetLang or "vi")
            out = [translator.translate(t) if t and t.strip() else "" for t in req.texts]
        return {"success": True, "translated": out}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 4. Giải phóng cổng 8000 & Bật Server Uvicorn trong luồng Daemon
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(1)
import uvicorn
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="error")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)

# 5. Mở Cổng Public (Auto Ngrok / Cloudflare Tunnel / LocalTunnel)
public_url = None

if NGROK_AUTH_TOKEN and len(NGROK_AUTH_TOKEN.strip()) > 10:
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
        ngrok.kill()
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
    except Exception as ne:
        print(f"⚠️ Ngrok note: {ne}")

if not public_url:
    print("⏳ Đang tạo cổng kết nối Cloudflare Tunnel miễn phí (Không cần authtoken)...")
    try:
        os.system("curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared")
        os.system("chmod +x /tmp/cloudflared")
        cf_log = open("/tmp/cf.log", "w")
        subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=cf_log, stderr=cf_log)
        time.sleep(3)
        for _ in range(12):
            if os.path.exists("/tmp/cf.log"):
                content = open("/tmp/cf.log").read()
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if m:
                    public_url = m.group(0)
                    break
            time.sleep(1)
    except Exception as ce:
        print(f"⚠️ Cloudflare note: {ce}")

if not public_url:
    print("⏳ Đang chuyển sang cổng LocalTunnel...")
    try:
        lt_log = open("/tmp/lt.log", "w")
        subprocess.Popen(["npx", "localtunnel", "--port", "8000"], stdout=lt_log, stderr=lt_log)
        time.sleep(4)
        if os.path.exists("/tmp/lt.log"):
            m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", open("/tmp/lt.log").read())
            if m: public_url = m.group(0)
    except Exception as le:
        print(f"⚠️ LocalTunnel note: {le}")

print("="*65)
print(f"🎉 [4/4] SERVER COLAB AI (lama-manga-onnx) ĐÃ SẴN SÀNG KHỞI CHẠY!")
if public_url:
    print(f"🔗 COPY URL NÀY DÁN VÀO WEB APP / APK: {public_url}")
else:
    print("🔗 Mở cổng local: http://localhost:8000")
print("="*65)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Server đã dừng.")

